# Explainable AI for Medical Text Classification
## Comparing LIME and Integrated Gradients on Medical Transcriptions
**Course:** Explainable AI (SOW-BKI266) — Radboud University 2025-2026

This notebook demonstrates two XAI methods applied to a transformer-based medical text classifier:
- **LIME** (Local Interpretable Model-agnostic Explanations)
- **Integrated Gradients** (via the Captum library)

Dataset: [Medical Transcriptions (mtsamples.csv)](https://www.kaggle.com/datasets/tboyle10/medicaltranscriptions)

## 1. Install Required Libraries

In [ ]:
# Run this cell once to install all required libraries
!pip install transformers torch captum lime scikit-learn pandas numpy matplotlib seaborn

## 2. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    AdamW,
    get_linear_schedule_with_warmup
)

from lime.lime_text import LimeTextExplainer
from captum.attr import IntegratedGradients, LayerIntegratedGradients
from captum.attr import visualization as viz

# Set random seeds for reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 3. Load and Preprocess the Dataset

Make sure `mtsamples.csv` is in the same folder as this notebook.

In [ ]:
# Load dataset
df = pd.read_csv('mtsamples.csv', index_col=0)
print(f'Dataset shape: {df.shape}')
df.head()

In [ ]:
# Inspect columns
print(df.columns.tolist())
print(df['medical_specialty'].value_counts())

In [ ]:
# Keep only the top 5 most frequent specialties for simplicity
TOP_N = 5
top_specialties = df['medical_specialty'].value_counts().head(TOP_N).index.tolist()
df = df[df['medical_specialty'].isin(top_specialties)].copy()

# Use transcription text as input
df = df[['transcription', 'medical_specialty']].dropna()
df.columns = ['text', 'label']

# Truncate text to 512 characters for speed
df['text'] = df['text'].str.strip().str[:512]

print(f'Filtered dataset shape: {df.shape}')
print(df['label'].value_counts())

In [ ]:
# Encode labels
le = LabelEncoder()
df['label_enc'] = le.fit_transform(df['label'])
NUM_CLASSES = len(le.classes_)
print(f'Classes: {le.classes_}')
print(f'Number of classes: {NUM_CLASSES}')

In [ ]:
# Plot class distribution
plt.figure(figsize=(8, 4))
sns.countplot(y='label', data=df, order=df['label'].value_counts().index)
plt.title('Class Distribution (Top 5 Medical Specialties)')
plt.xlabel('Count')
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150)
plt.show()

In [ ]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    df['text'].tolist(),
    df['label_enc'].tolist(),
    test_size=0.2,
    random_state=SEED,
    stratify=df['label_enc'].tolist()
)
print(f'Train size: {len(X_train)}, Test size: {len(X_test)}')

## 4. Tokenization and Dataset

In [ ]:
MODEL_NAME = 'distilbert-base-uncased'
tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)

class MedicalDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'label': torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_dataset = MedicalDataset(X_train, y_train, tokenizer)
test_dataset  = MedicalDataset(X_test,  y_test,  tokenizer)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=16, shuffle=False)

print('Datasets ready.')

## 5. Fine-tune DistilBERT

In [ ]:
model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=NUM_CLASSES
).to(device)

EPOCHS = 3
optimizer = AdamW(model.parameters(), lr=2e-5)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=0, num_training_steps=total_steps
)

print(f'Model loaded. Training for {EPOCHS} epochs...')

In [ ]:
# Training loop
train_losses = []

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for batch in train_loader:
        optimizer.zero_grad()
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['label'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        total_loss += loss.item()

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

    avg_loss = total_loss / len(train_loader)
    train_losses.append(avg_loss)
    print(f'Epoch {epoch+1}/{EPOCHS} — Loss: {avg_loss:.4f}')

print('Training complete!')

In [ ]:
# Plot training loss
plt.figure(figsize=(6, 4))
plt.plot(range(1, EPOCHS+1), train_losses, marker='o')
plt.title('Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.tight_layout()
plt.savefig('training_loss.png', dpi=150)
plt.show()

## 6. Evaluate the Model

In [ ]:
model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for batch in test_loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['label'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print(classification_report(all_labels, all_preds, target_names=le.classes_))

## 7. XAI Method 1 — LIME

In [ ]:
# LIME prediction function — takes a list of texts, returns probability array
def lime_predict(texts):
    model.eval()
    encodings = tokenizer(
        texts,
        max_length=256,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    ).to(device)
    with torch.no_grad():
        outputs = model(**encodings)
    probs = F.softmax(outputs.logits, dim=1).cpu().numpy()
    return probs

# Initialize LIME explainer
lime_explainer = LimeTextExplainer(class_names=le.classes_)

# Pick a sample from test set
SAMPLE_IDX = 0
sample_text  = X_test[SAMPLE_IDX]
sample_label = y_test[SAMPLE_IDX]

print(f'Sample text (first 300 chars):\n{sample_text[:300]}')
print(f'\nTrue label: {le.classes_[sample_label]}')

In [ ]:
# Generate LIME explanation
lime_exp = lime_explainer.explain_instance(
    sample_text,
    lime_predict,
    num_features=10,
    num_samples=500,
    labels=[sample_label]
)

print(f'LIME explanation for class: {le.classes_[sample_label]}')
lime_exp.show_in_notebook(text=True, labels=(sample_label,))

In [ ]:
# Save LIME explanation as a figure
fig = lime_exp.as_pyplot_figure(label=sample_label)
fig.suptitle(f'LIME — Top features for class: {le.classes_[sample_label]}', fontsize=12)
plt.tight_layout()
plt.savefig('lime_explanation.png', dpi=150)
plt.show()
print('LIME explanation saved.')

## 8. XAI Method 2 — Integrated Gradients

In [ ]:
# Wrapper model for Captum — takes input_ids and attention_mask, returns logits
class ModelWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, input_ids, attention_mask):
        output = self.model(input_ids=input_ids, attention_mask=attention_mask)
        return output.logits

wrapped_model = ModelWrapper(model).to(device)
wrapped_model.eval()

# Integrated Gradients operates on embeddings
ig = LayerIntegratedGradients(wrapped_model, wrapped_model.model.distilbert.embeddings)

print('Integrated Gradients setup complete.')

In [ ]:
# Tokenize the sample
encoding = tokenizer(
    sample_text,
    max_length=256,
    padding='max_length',
    truncation=True,
    return_tensors='pt'
).to(device)

input_ids      = encoding['input_ids']
attention_mask = encoding['attention_mask']

# Baseline: all [PAD] tokens
baseline_ids = torch.zeros_like(input_ids).to(device)

# Compute attributions
attributions, delta = ig.attribute(
    inputs=input_ids,
    baselines=baseline_ids,
    additional_forward_args=(attention_mask,),
    target=sample_label,
    return_convergence_delta=True,
    n_steps=50
)

print(f'Convergence delta: {delta.item():.4f} (closer to 0 is better)')

In [ ]:
# Summarize attributions per token
attributions_sum = attributions.sum(dim=-1).squeeze(0)
attributions_sum = attributions_sum / torch.norm(attributions_sum)  # normalize
attributions_sum = attributions_sum.cpu().detach().numpy()

# Get tokens
tokens = tokenizer.convert_ids_to_tokens(input_ids.squeeze().cpu().numpy())

# Remove padding tokens
non_pad = [(t, a) for t, a in zip(tokens, attributions_sum) if t != '[PAD]']
tokens_clean, attrs_clean = zip(*non_pad)

# Show top 15 most influential tokens
token_attr = sorted(zip(tokens_clean, attrs_clean), key=lambda x: abs(x[1]), reverse=True)[:15]
top_tokens, top_attrs = zip(*token_attr)

print('Top 15 tokens by Integrated Gradients attribution:')
for t, a in zip(top_tokens, top_attrs):
    print(f'  {t:20s}: {a:.4f}')

In [ ]:
# Visualize Integrated Gradients attributions
colors = ['tomato' if a < 0 else 'steelblue' for a in top_attrs]

plt.figure(figsize=(10, 5))
bars = plt.barh(top_tokens[::-1], top_attrs[::-1], color=colors[::-1])
plt.axvline(x=0, color='black', linewidth=0.8)
plt.title(f'Integrated Gradients — Top tokens for class: {le.classes_[sample_label]}')
plt.xlabel('Attribution Score (normalized)')
plt.tight_layout()
plt.savefig('integrated_gradients_explanation.png', dpi=150)
plt.show()
print('Integrated Gradients explanation saved.')

## 9. Comparison: LIME vs Integrated Gradients

In [ ]:
# Get LIME top words
lime_words = dict(lime_exp.as_list(label=sample_label))

# Get IG top words (clean subword tokens)
ig_words = {t.replace('##', ''): a for t, a in zip(top_tokens, top_attrs)}

# Find overlapping words
overlap = set(lime_words.keys()) & set(ig_words.keys())
print(f'Overlapping important words between LIME and IG: {overlap if overlap else "None"}')

# Side-by-side comparison plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# LIME
lime_items = sorted(lime_words.items(), key=lambda x: abs(x[1]), reverse=True)[:10]
l_words, l_vals = zip(*lime_items)
l_colors = ['tomato' if v < 0 else 'steelblue' for v in l_vals]
axes[0].barh(l_words[::-1], l_vals[::-1], color=l_colors[::-1])
axes[0].axvline(x=0, color='black', linewidth=0.8)
axes[0].set_title('LIME — Top 10 Words')
axes[0].set_xlabel('Importance Score')

# IG
ig_items = sorted(ig_words.items(), key=lambda x: abs(x[1]), reverse=True)[:10]
ig_w, ig_v = zip(*ig_items)
ig_colors = ['tomato' if v < 0 else 'steelblue' for v in ig_v]
axes[1].barh(ig_w[::-1], ig_v[::-1], color=ig_colors[::-1])
axes[1].axvline(x=0, color='black', linewidth=0.8)
axes[1].set_title('Integrated Gradients — Top 10 Tokens')
axes[1].set_xlabel('Attribution Score (normalized)')

plt.suptitle(f'LIME vs Integrated Gradients — Class: {le.classes_[sample_label]}', fontsize=13)
plt.tight_layout()
plt.savefig('comparison_lime_vs_ig.png', dpi=150)
plt.show()
print('Comparison plot saved.')

## 10. Faithfulness Evaluation — Deletion Test

We evaluate faithfulness by progressively removing the most important words identified by each method and measuring the drop in model confidence. A steeper drop indicates a more faithful explanation.

In [ ]:
def deletion_test(text, important_words, predict_fn, label_idx, steps=10):
    """
    Progressively delete top important words and measure confidence drop.
    Returns list of confidence scores at each deletion step.
    """
    words = text.split()
    sorted_words = sorted(important_words.items(), key=lambda x: abs(x[1]), reverse=True)
    top_words = [w for w, _ in sorted_words[:steps]]

    scores = []
    current_text = text

    # Baseline confidence
    prob = predict_fn([current_text])[0][label_idx]
    scores.append(prob)

    for word in top_words:
        current_text = current_text.replace(word, '[MASK]')
        prob = predict_fn([current_text])[0][label_idx]
        scores.append(prob)

    return scores

# Run deletion test for LIME
lime_deletion = deletion_test(sample_text, lime_words, lime_predict, sample_label)

# Run deletion test for IG
ig_deletion = deletion_test(sample_text, ig_words, lime_predict, sample_label)

# Plot
plt.figure(figsize=(8, 4))
plt.plot(lime_deletion, marker='o', label='LIME', color='steelblue')
plt.plot(ig_deletion,   marker='s', label='Integrated Gradients', color='tomato')
plt.title('Faithfulness Evaluation — Deletion Test')
plt.xlabel('Number of Words Deleted')
plt.ylabel(f'Model Confidence (class: {le.classes_[sample_label]})')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('faithfulness_deletion_test.png', dpi=150)
plt.show()
print('Faithfulness evaluation saved.')

## 11. Summary

| Aspect | LIME | Integrated Gradients |
|---|---|---|
| Type | Model-agnostic, post-hoc | Model-specific, gradient-based |
| Input | Raw text (word-level) | Tokenized input (subword-level) |
| Speed | Slower (requires many forward passes) | Faster (single backward pass) |
| Faithfulness | Approximate (surrogate model) | Exact (axiomatic guarantees) |
| Interpretability | High (word-level, human-readable) | Medium (subword tokens) |
| Best for | Non-technical stakeholders | Technical users / researchers |

**Key takeaway:** LIME produces more human-readable explanations while Integrated Gradients provides stronger theoretical faithfulness guarantees. In a medical setting, both are complementary — LIME for clinician-facing explanations, IG for model auditing.